# 08 · Paired statistics for all experiments

No training. Reads the `*_by_seed.csv` files and compares each condition with its experiment's reference across the five matched splits:

* mean paired difference (condition minus reference) and its *t*-based 95% CI,
* paired *t*-test, Holm-adjusted within each family and separately for each metric,
* exact Wilcoxon signed-rank *p* for completeness. With five pairs its smallest possible two-sided value is 0.0625, so it is never used to claim significance.

The five splits overlap, so these numbers describe variation between splits of one dataset, not generalisation to new data. The families below match the ones described in the manuscript; if you change one here, change the Methods text too.

| | |
|---|---|
| Trains models | no |
| Needs | notebooks 01-07 |
| Writes | `results/stats/paired_tests.csv` (every comparison), `paired_tests_key.csv` (F1, PR-AUC, ROC-AUC only) |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import numpy as np, pandas as pd

METRICS = ["F1", "PR_AUC", "ROC_AUC", "MCC", "Precision", "Recall"]
shares = cfg["e6"]["fraud_shares"]

# family, experiment folder, file, condition column, reference, compared conditions, row filter
FAMILIES = [
    ("E1 components", "E1_components", "runs_by_seed.csv", "Experiment", "RAREGuard_Full",
     ["IF_Only", "OCSVM_Only", "AE_Only", "RAREGuard_No_FS", "RAREGuard_No_PCA"], {"Threshold_Mode": "Fully unsupervised"}),
    ("E2 fusion baselines", "E2_E3_fusion", "fusion_baselines_by_seed.csv", "Configuration", "RAREGuard_30_30_40",
     ["Equal_3", "OCSVM_AE_50_50", "Median_3", "IF_OCSVM_50_50", "OCSVM_Only", "Rank_Mean_3", "IF_AE_50_50", "IF_Only", "AE_Only"], None),
    ("E3 fusion weights", "E2_E3_fusion", "fusion_weights_by_seed.csv", "Configuration", "Original_30_30_40",
     ["Equal_Weights", "IF_Dominant", "OCSVM_Dominant", "AE_Dominant"], None),
    ("E4 threshold estimators", "E4_E5_thresholds", "threshold_methods_by_seed.csv", "Method", "Percentile",
     ["EVT_GPD", "MAD", "IQR"], None),
    ("E4 label-assisted reference", "E4_E5_thresholds", "threshold_methods_by_seed.csv", "Method", "Percentile",
     ["Label_Assisted"], None),
    ("E6 reference contamination", "E6_reference_contamination", "reference_contamination_by_seed.csv",
     "Training_Reference_Contamination", shares[0], shares[1:], None),
    ("E7 early stopping", "E7_early_stopping", "early_stopping_by_seed.csv", "Protocol", "Shared_Validation",
     ["Heldout_Legitimate_10pct"], None),
    ("E9 PaySim representation", "E9_paysim", "representation_by_seed.csv", "Representation", "Baseline", ["Enhanced"], None),
    ("E9 PaySim PCA", "E9_paysim", "pca_ablation_by_seed.csv", "PCA", "On", ["Off"], None),
]

run = rg.Run(cfg, "stats", "08_statistics.ipynb", "Paired t-tests with Holm correction for all families")
tables = []
for family, exp, file, col, ref, others, keep in FAMILIES:
    path = rg.proj(cfg, "results", exp, file)
    if not os.path.exists(path):
        print(f"skipped {family}: {exp}/{file} not found")
        continue
    df = pd.read_csv(path)
    for k, v in (keep or {}).items():
        df = df[df[k] == v]
    tables.append(rg.compare_family(df, col, ref, others, METRICS, family).assign(Source=f"{exp}/{file}"))

tests = pd.concat(tables, ignore_index=True)
run.save_csv(tests, "paired_tests.csv")
key = tests[tests.Metric.isin(["F1", "PR_AUC", "ROC_AUC"])]
run.save_csv(key, "paired_tests_key.csv")
run.finish()

In [ ]:
show = key.assign(CI=lambda d: d.apply(lambda r: f"[{r.CI95_Low:+.3f}, {r.CI95_High:+.3f}]", axis=1))
show = show[["Family", "Metric", "Condition", "Reference_Mean", "Other_Mean", "Delta", "CI", "Seeds_Higher", "p_Holm"]]
for fam, block in show.groupby("Family", sort=False):
    print("\n" + fam)
    display(block.drop(columns="Family").round(4))